# Genie AFTER: Analyst Questions on the Enriched Catalog

**Run on serverless compute** (no Neo4j Spark Connector required).
**Prerequisite:** `04_pull_gold_tables` must have completed — the three gold
tables must exist in Unity Catalog.

This notebook asks five analyst questions to the AFTER Genie Space — the space
pointed at the enriched gold tables. Each question comes from one of five
categories that the enriched catalog unlocks. The structural dimensions written
by the GDS pipeline (`community_id`, `fraud_risk_tier`, `similarity_score`) are
now scalar columns. Genie reads them the same way it reads any other warehouse
column.

The structural questions from `01_genie_before` are not re-run here. The
contrast is not "same question, better answer." It is "the enriched catalog
makes a new class of analyst question answerable."

In [ ]:
%pip install --upgrade databricks-sdk --quiet
dbutils.library.restartPython()

In [ ]:
CATALOG  = "graph-enriched-lakehouse"
SCHEMA   = "graph-enriched-schema"

SPACE_ID = dbutils.secrets.get("neo4j-graph-engineering", "genie_space_id_after")

GOLD_FEATURES_TABLE    = f"{CATALOG}.{SCHEMA}.gold_accounts"
GOLD_PAIRS_TABLE       = f"{CATALOG}.{SCHEMA}.gold_account_similarity_pairs"
GOLD_COMMUNITIES_TABLE = f"{CATALOG}.{SCHEMA}.gold_fraud_ring_communities"

In [ ]:
from databricks.sdk import WorkspaceClient
from demo_utils import genie_caller

w = WorkspaceClient()
print("Connected to:", w.config.host)
ask_genie = genie_caller(w, SPACE_ID)

def _status(r):
    if r["df"] is not None and len(r["df"]) > 0:
        return "RESPONDED"
    elif r["df"] is not None:
        return "NO DATA"
    return "NO DATA" if r["text"] else "ERROR"

In [ ]:
print("Pre-flight: verifying gold tables exist...")
all_present = True
for tbl in [GOLD_FEATURES_TABLE, GOLD_PAIRS_TABLE, GOLD_COMMUNITIES_TABLE]:
    try:
        w.tables.get(full_name=tbl)
        print(f"  [OK]      {tbl}")
    except Exception:
        print(f"  [MISSING] {tbl}  — run 04_pull_gold_tables first")
        all_present = False

if not all_present:
    dbutils.notebook.exit("One or more gold tables are missing. Run 04_pull_gold_tables before continuing.")

## Category 1 — Portfolio Composition

With `community_id` and `is_ring_community` as scalar columns, Genie can answer
portfolio-composition questions that were structurally unavailable before
enrichment. The question below is the same one labeled
`NOT AVAILABLE ON THIS CATALOG` in `01_genie_before` — now it resolves.

In [ ]:
CAT1_Q = (
    "What share of accounts sits in communities flagged as ring candidates, "
    "broken out by region?"
)

print("[1] portfolio_ring_share_by_region — asking...")
r1 = ask_genie(CAT1_Q)

print(f"[1] portfolio_ring_share_by_region — {_status(r1)}")
if r1["df"] is not None:
    print(f"    Rows: {len(r1['df'])}")
    print(f"    SQL:  {r1['sql']}")
    display(r1["df"])
elif r1["text"]:
    print(f"    Text: {r1['text'][:400]}")

## Category 2 — Cohort Comparisons

`fraud_risk_tier` partitions accounts into `high` and `low` cohorts based on
their PageRank centrality. Standard BI cohort comparisons — spending mix,
transfer behavior, score distributions — now have a structurally-defined cohort
alongside the demographic and transactional segments that were always available.

In [ ]:
CAT2_Q = (
    "How does merchant-category spending mix differ between ring-community "
    "accounts and the baseline?"
)

print("[2] cohort_merchant_spend_mix — asking...")
r2 = ask_genie(CAT2_Q)

print(f"[2] cohort_merchant_spend_mix — {_status(r2)}")
if r2["df"] is not None:
    print(f"    Rows: {len(r2['df'])}")
    print(f"    SQL:  {r2['sql']}")
    display(r2["df"])
elif r2["text"]:
    print(f"    Text: {r2['text'][:400]}")

## Category 3 — Community Rollups

`gold_fraud_ring_communities` provides pre-aggregated ring-candidate community
summaries. Rollup questions over the already-labeled set — total balance held,
regional concentration, internal vs. external transfer ratios — are the same BI
rollup questions a business stakeholder would ask about any pre-defined segment.

In [ ]:
CAT3_Q = (
    "For ring-candidate communities taken together, what is the total balance "
    "held by their members and what share of the book do they represent?"
)

print("[3] rollup_ring_candidate_total_balance — asking...")
r3 = ask_genie(CAT3_Q)

print(f"[3] rollup_ring_candidate_total_balance — {_status(r3)}")
if r3["df"] is not None:
    print(f"    Rows: {len(r3['df'])}")
    print(f"    SQL:  {r3['sql']}")
    display(r3["df"])
elif r3["text"]:
    print(f"    Text: {r3['text'][:400]}")

## Category 4 — Operational Workload

Once `fraud_risk_tier` is a column, operations teams can ask capacity and queue
questions about the investigation workload the structural signal implies —
questions that previously had no structural handle at all.

In [ ]:
CAT4_Q = (
    "How many accounts would need investigator review if the bar is high "
    "risk tier, and what is the regional breakdown of that workload?"
)

print("[4] operational_review_queue_size — asking...")
r4 = ask_genie(CAT4_Q)

print(f"[4] operational_review_queue_size — {_status(r4)}")
if r4["df"] is not None:
    print(f"    Rows: {len(r4['df'])}")
    print(f"    SQL:  {r4['sql']}")
    display(r4["df"])
elif r4["text"]:
    print(f"    Text: {r4['text'][:400]}")

## Category 5 — Merchant-Side Analysis

Merchants were always in the catalog, but the base tables had no way to filter
them by the structural cohort they serve. With `community_id` and
`fraud_risk_tier` as columns on `gold_accounts`, merchant questions can now be
asked conditionally on community membership or risk tier.

In [ ]:
CAT5_Q = (
    "Which merchants are most commonly visited by accounts in "
    "ring-candidate communities?"
)

print("[5] merchant_ring_community_favorites — asking...")
r5 = ask_genie(CAT5_Q)

print(f"[5] merchant_ring_community_favorites — {_status(r5)}")
if r5["df"] is not None:
    print(f"    Rows: {len(r5['df'])}")
    print(f"    SQL:  {r5['sql']}")
    display(r5["df"])
elif r5["text"]:
    print(f"    Text: {r5['text'][:400]}")

In [ ]:
results = [
    ("portfolio_ring_share_by_region",      r1),
    ("cohort_merchant_spend_mix",           r2),
    ("rollup_ring_candidate_total_balance", r3),
    ("operational_review_queue_size",       r4),
    ("merchant_ring_community_favorites",   r5),
]

print("=" * 78)
print("Genie AFTER — Summary")
print("=" * 78)
print()
print("Purpose: demonstrate analyst questions unlocked by the enriched catalog")
print()
responded = sum(1 for _, r in results if _status(r) == "RESPONDED")
for name, r in results:
    print(f"  {_status(r):<12}  {name}")
print()
print("-" * 78)
print(f"Responded: {responded}/{len(results)}")
print()
print("The structural dimensions — community_id, fraud_risk_tier, similarity_score —")
print("are scalar columns. Genie answers these analyst questions the same way it")
print("answers questions against any other warehouse table.")
print()
print("Same Databricks spend. Strictly more answers.")

## What's Next

The enrichment pipeline is complete and the Genie AFTER run is done.

To see ML lift from the same graph features — a gradient-boosting classifier
trained on tabular features versus one trained on all features including
`risk_score`, `community_id`, and `similarity_score` — open
**`06_train_model`** (optional).